# nb58 - Quantile-width stack: qd loss + EMA composition (H22)

**Error analysis / context.** Two independently adopted wins have never been combined: the coverage-width (qd) objective won against its like-for-like anchor WITHOUT EMA (nb54: 0.0418/0.0418 vs nb44 0.0442 +/- 0.0005 = -0.0024, past the 0.002 criterion), and EMA won against the same anchor (nb52: 0.0424 +/- 0.0003). If the mechanisms are independent (loss shape vs weight averaging), they should compound.

**Question.** Does qd + EMA compound, and does the resulting 5-seed stack (TTA + joint direct calibration) beat the 0.0409 record?

**Hypothesis.** H22: qd+EMA singles land at or below 0.0415; the full stack lands below 0.0409 (LS-calib number also reported per the anti-overfit guardrail).

**Proof criterion.** 5 seeds; singles compared against BOTH anchors (nb54 qd 0.0418, nb52 EMA 0.0424); stack judged against 0.0409 (direct) / 0.0415 (LS). Hyperparameters identical to nb54 qd (lambda 0.5, tau 0.02) and nb52 EMA (decay 0.999) - nothing new to tune.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, prep
from picocal_models import SubNetFQ, QUANTILES, width_binned_calibration, CFG
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB58_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB58_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
D = prep(4, ME, CE, ng=6)
T = dict(X=torch.from_numpy(D['X']).to(DEVICE), M=torch.from_numpy(D['M']).to(DEVICE),
         G=torch.from_numpy(D['G']).to(DEVICE), Y=torch.from_numpy(D['y']).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(D['Eraw']).to(DEVICE))
ktr, kva, kte, ctr = D['ktr'], D['kva'], D['kte'], D['ctr']
y = D['y']; Et = D['Et']
QS = torch.tensor(QUANTILES, device=DEVICE)
print(f'device {DEVICE} | mode {MODE} | build+prep {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16
device cuda | mode full | build+prep 148s


In [2]:
LAM_QD = 0.5
TAU = 0.02
def qd_loss(q, yb):
    d = yb - q
    pin = torch.maximum(QS * d, (QS - 1) * d).mean()
    width = (q[:, 2] - q[:, 1]).abs() + (q[:, 1] - q[:, 0]).abs()
    inside = torch.sigmoid((yb.squeeze(1) - q[:, 0]) / TAU) * torch.sigmoid((q[:, 2] - yb.squeeze(1)) / TAU)
    cov_pen = torch.relu(0.5 - inside.mean()) ** 2
    return pin + LAM_QD * (width.mean() + 10.0 * cov_pen)
def train_eval(seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr_idx = np.concatenate([np.asarray(ktr), ctr])
    ck = CKPT / f'nb58_qdema_s{seed}.pt'
    def batches(idx, bs, sh=None):
        idx = np.asarray(idx)
        if sh is not None: idx = sh.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(m_, b): return m_(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def vloss(m_):
        m_.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256):
                d = T['Y'][b] - fwd(m_, b)
                s += torch.maximum(QS * d, (QS - 1) * d).mean().item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); ema.load_state_dict(st['ema'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume s{seed} from ep {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], rng):
            opt.zero_grad()
            qd_loss(fwd(model, b), T['Y'][b]).backward()
            opt.step()
            ema.update_parameters(model)
        sched.step()
        vv = vloss(ema.module)
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(ema.module.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(),
                        sched=sched.state_dict(), best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    final = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    final.load_state_dict(bstate); final.eval()
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 256): out.append(fwd(final, b).cpu().numpy())
        return np.concatenate(out)
    pe = width_binned_calibration(run(kva), run(kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

In [3]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1, 2, 3, 4]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb58_qdema{TAG}.csv'
done = set()
if CSVP.exists():
    done = set(pd.read_csv(CSVP)['seed'])
    print('resume, done:', sorted(done))
for seed in SEEDS:
    if seed in done: print('skip', seed); continue
    t1 = time.time()
    sig, pe = train_eval(seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb58_pred{TAG}_qdema_s{seed}.npy', pe)
    row = dict(seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'qdema seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

qdema seed 0: sigma_eff 0.0413 (2529s)


qdema seed 1: sigma_eff 0.0417 (2515s)


qdema seed 2: sigma_eff 0.0412 (2532s)


qdema seed 3: sigma_eff 0.0426 (2465s)


qdema seed 4: sigma_eff 0.0421 (3084s)


 seed  sigma_eff  elapsed
    0     0.0413     2529
    1     0.0417     2515
    2     0.0412     2532
    3     0.0426     2465
    4     0.0421     3084


## Full stack: qd+EMA x5 + D4 TTA + calibration (LS and direct both reported)

In [4]:
from scipy.optimize import minimize
DI, DJ = 3, 4
meanT = torch.tensor(D['mean'], device=DEVICE); stdT = torch.tensor(D['std'], device=DEVICE)
def tta_views(xb, mb):
    di = xb[:, :, DI] * stdT[DI] + meanT[DI]; dj = xb[:, :, DJ] * stdT[DJ] + meanT[DJ]
    for swap in (False, True):
        for s1 in (1.0, -1.0):
            for s2 in (1.0, -1.0):
                a = (dj if swap else di) * s1; b2 = (di if swap else dj) * s2
                xv = xb.clone()
                xv[:, :, DI] = torch.where(mb, (a - meanT[DI]) / stdT[DI], torch.zeros_like(a))
                xv[:, :, DJ] = torch.where(mb, (b2 - meanT[DJ]) / stdT[DJ], torch.zeros_like(b2))
                yield xv
MODELS = []
for s in range(5):
    ck = CKPT / f'nb58_qdema_s{s}.pt'
    if not ck.exists(): continue
    m = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    m.load_state_dict(torch.load(ck, map_location=DEVICE)['bstate']); m.eval()
    MODELS.append(m)
print('loaded', len(MODELS), 'qd+EMA models')
def infer(idx):
    per = []
    with torch.no_grad():
        for mdl in MODELS:
            out = []
            for j in range(0, len(idx), 256):
                b = torch.from_numpy(np.asarray(idx[j:j+256])).to(DEVICE)
                xb, mb2 = T['X'][b], T['M'][b]
                qs = [mdl(xv, mb2, T['G'][b], T['E'][b]) for xv in tta_views(xb, mb2)]
                out.append(torch.stack(qs).mean(0).cpu().numpy())
            per.append(np.concatenate(out))
    return np.stack(per).mean(0)
if MODE == 'full' and len(MODELS) >= 2:
    qv = infer(kva); qt = infer(kte)
    yva = y[kva]; Ev = np.exp(yva); te_e = Et[kte]
    wv = qv[:, 2] - qv[:, 0]; wt_ = qt[:, 2] - qt[:, 0]
    cuts = np.quantile(wv, [1/3, 2/3])
    gv = np.digitize(wv, cuts); gt = np.digitize(wt_, cuts)
    p0 = []
    for g in range(3):
        a0, b0 = np.polyfit(qv[gv == g, 1], yva[gv == g], 1)
        p0 += [a0, b0]
    def apply(p, q, grp):
        pe = np.empty(len(q))
        for g in range(3):
            pe[grp == g] = np.exp(p[2*g] * q[grp == g, 1] + p[2*g+1])
        return pe
    def obj(p): return resolution(apply(p, qv, gv), Ev)['sigma_eff']
    res = minimize(obj, p0, method='Nelder-Mead', options=dict(xatol=1e-5, fatol=1e-7, maxiter=3000))
    p = res.x if res.fun <= obj(np.array(p0)) else np.array(p0)
    pe_ls = apply(np.array(p0), qt, gt)
    pe_dir = apply(p, qt, gt)
    np.save(OUT / 'nb58_pred_stack.npy', pe_dir)
    print(f'qdEMA x{len(MODELS)} + TTA (LS calib):     {resolution(pe_ls, te_e)["sigma_eff"]:.4f}   [record LS 0.0415]')
    print(f'qdEMA x{len(MODELS)} + TTA + direct calib: {resolution(pe_dir, te_e)["sigma_eff"]:.4f}   [record 0.0409 | targets 0.06/0.045/0.035/0.032/0.030/0.030]')
    edges = np.quantile(te_e, np.linspace(0, 1, 7))
    bins = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        bins.append(f'{resolution(pe_dir[mm], te_e[mm])["sigma_eff"]:.4f}')
    print('per-bin ' + ' / '.join(bins))

loaded 5 qd+EMA models


qdEMA x5 + TTA (LS calib):     0.0411   [record LS 0.0415]
qdEMA x5 + TTA + direct calib: 0.0402   [record 0.0409 | targets 0.06/0.045/0.035/0.032/0.030/0.030]
per-bin 0.0625 / 0.0447 / 0.0336 / 0.0336 / 0.0326 / 0.0331
